# Phase 2 — Section 2 Report
## EDA, Leakage-Safe Preparation, and a Hybrid Similar-Question Recommender

**Dataset:** Crawled Stack Overflow C++ Questions Dataset  
**Final task:** Semantic similar-question recommendation

این گزارش، Section 2 را به شکل یک کار واقعی Data Science ارائه می‌دهد: تحلیل اکتشافی، آماده‌سازی بدون **data leakage**، و ارزیابی کمّی مدل با متریک‌های حرفه‌ای retrieval (`Hit@K`, `Recall@K`, `MAP@K`, `MRR@K`, `nDCG@K`).

> منطق مدل‌سازی در `scripts/preprocess.py` و `scripts/feature_engineering.py` پیاده شده و توسط `pipeline.py` اجرا می‌شود. این نوت‌بوک همان نتایج تولیدشده توسط pipeline را بارگذاری و تفسیر می‌کند تا گزارش و کد دقیقاً هم‌خوان بمانند (بدون آموزش دوباره‌ی مدل داخل گزارش).

## 1. Strategy summary

تصمیم‌های اصلی این بخش:

1. **Temporal split** به‌جای split تصادفی: سؤال‌های جدیدتر به validation/test می‌روند؛ این دقیقاً سناریوی دنیای واقعی است (از سؤال‌های قدیمی به سؤال جدید پیشنهاد می‌دهیم).
2. **Vectorizerها فقط روی train فیت می‌شوند**؛ validation/test فقط transform می‌شوند.
3. چون دیتاست برچسب «تکراری بودن» انسانی ندارد، از **هم‌پوشانی تگ** به‌عنوان سیگنال ضعیف relevance استفاده می‌کنیم؛ تگ عمومی `c++` از این سیگنال حذف می‌شود تا relevance مصنوعی نسازد.
4. **چهار** نمایش متنی مقایسه و در یک **ensemble وزنی** ترکیب می‌شوند:
   - TF-IDF روی عنوان (word)
   - TF-IDF روی سند (word)
   - TF-IDF کاراکتری (`char_wb`) برای syntax و نام توابع/کتابخانه‌های C++
   - **LSA** (`TruncatedSVD` روی TF-IDF سند) — یک نمایش معناییِ متراکم که هم‌رخدادی و مترادف‌ها را فراتر از تطابق لغوی می‌گیرد؛ یک «امبدینگ» سبک بدون dependency اضافه.
5. وزن‌ها فقط روی validation انتخاب می‌شوند و test فقط یک‌بار برای گزارش نهایی استفاده می‌شود (ضد overfitting).

> **یادداشت صادقانه درباره‌ی LSA:** چون معیار ارزیابی ما هم‌پوشانی تگ (سیگنالی لغوی/موضوعی) است، نمایش‌های TF-IDF که دقیقاً همان تطابق لغوی را می‌گیرند در این متریک قوی‌اند؛ بنابراین LSA با وزن کمی انتخاب می‌شود و لیفت بزرگی در این متریک نمی‌دهد. ارزش واقعی LSA — یافتن سؤالات هم‌معنا ولی با کلمات متفاوت — را این برچسب ضعیف کامل نشان نمی‌دهد. LSA را به‌خاطر **تنوع نمایش** و به‌عنوان مؤلفه‌ی واقعاً معنایی برای فاز ۳ نگه می‌داریم.

In [1]:
from pathlib import Path
import json, os, sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
env_root = Path(os.environ['PROJECT_ROOT']).resolve() if os.environ.get('PROJECT_ROOT') else None
candidates = [p for p in [env_root, cwd, *cwd.parents, cwd / 'Phase 2' / 'Project_P2'] if p is not None]
PROJECT_ROOT = next((p for p in candidates if (p / 'pipeline.py').exists() and (p / 'requirements.txt').exists()), cwd)
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.database_connection import get_database_path
from scripts.load_data import load_question_dataframe

REPORT_DIR = PROJECT_ROOT / 'data' / 'reports'
if not get_database_path().exists():
    raise FileNotFoundError('Database not found. Run `python pipeline.py` first.')

questions = load_question_dataframe()
questions['creation_at'] = pd.to_datetime(questions['creation_at'], utc=True, errors='coerce')
questions['tag_list'] = questions['tags'].fillna('').str.split('|')
print(f'Project root: {PROJECT_ROOT}')
print(f'Questions loaded from SQLite: {len(questions):,}')

Project root: /home/user/Data-Science/Phase 2/Project_P2
Questions loaded from SQLite: 2,500


## 2. EDA and data-quality report

ساختار داده، missingها، توزیع‌ها و outlierها را بررسی می‌کنیم تا هم ما و هم منتور تصویر روشنی از دیتاست داشته باشیم.

In [2]:
quality = pd.DataFrame({
    'check': ['rows', 'unique_question_id', 'missing_title', 'missing_body', 'blank_title',
              'questions_without_tags', 'min_creation', 'max_creation'],
    'value': [len(questions), questions['question_id'].nunique(),
              int(questions['title'].isna().sum()), int(questions['body_html'].isna().sum()),
              int(questions['title'].fillna('').str.strip().eq('').sum()),
              int(questions['tag_count'].eq(0).sum()),
              str(questions['creation_at'].min()), str(questions['creation_at'].max())]})
display(quality)
display(questions[['view_count','answer_count','score','tag_count']]
        .describe(percentiles=[.01,.25,.5,.75,.95,.99]).T)

,check,value
0,rows,2500
1,unique_question_id,2500
2,missing_title,0
3,missing_body,0
4,blank_title,0
5,questions_without_tags,0
6,min_creation,2008-08-01 12:13:50+00:00
7,max_creation,2026-05-28 16:11:59+00:00


,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
view_count,2500.0,25127.6464,156232.265188,10.0,58.0,171.0,500.5,6067.75,89376.90,457964.26,5131807.0
answer_count,2500.0,3.5972,4.841521,0.0,0.0,1.0,2.0,4.00,13.00,25.00,52.0
score,2500.0,35.3344,563.175400,-25.0,-3.0,0.0,2.0,7.00,89.15,486.43,27530.0
tag_count,2500.0,3.4392,1.146311,1.0,1.0,3.0,3.0,4.00,5.00,5.00,5.0


In [3]:
# Skew before/after transforms motivates the log/signed-log features.
import numpy as np
skew = pd.DataFrame({
    'raw_skew': questions[['view_count','answer_count','score']].skew().round(2),
    'log_skew': [np.log1p(questions['view_count'].clip(lower=0)).skew().round(2),
                 np.log1p(questions['answer_count'].clip(lower=0)).skew().round(2),
                 (np.sign(questions['score'])*np.log1p(questions['score'].abs())).skew().round(2)]})
display(skew)

,raw_skew,log_skew
view_count,19.35,0.82
answer_count,3.26,0.69
score,46.72,1.09


In [4]:
tag_freq = (questions['tag_list'].explode().replace('', pd.NA).dropna()
            .value_counts().rename_axis('tag').reset_index(name='count'))
display(tag_freq.head(20))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
tag_freq.head(15).sort_values('count').plot.barh(x='tag', y='count', ax=axes[0,0], legend=False)
axes[0,0].set(title='Top 15 tags', xlabel='question count')
axes[0,1].hist(np.log1p(questions['view_count'].clip(lower=0)), bins=40, color='#377eb8')
axes[0,1].set(title='log1p(view_count)', xlabel='log1p(views)', ylabel='questions')
axes[1,0].hist(questions['tag_count'], bins=np.arange(0.5,6.5,1), color='#984ea3', rwidth=0.9)
axes[1,0].set(title='Tags per question', xlabel='tag count', ylabel='questions')
monthly = questions.set_index('creation_at').resample('ME').size()
axes[1,1].plot(monthly.index, monthly.values, color='#e41a1c', linewidth=1)
axes[1,1].set(title='Question creation over time', xlabel='month', ylabel='questions')
fig.tight_layout(); display(fig); plt.close(fig)

,tag,count
0,c++,2500
1,c,128
2,c++11,117
3,qt,105
4,windows,104
5,templates,102
6,cmake,101
7,language-lawyer,98
8,winapi,89
9,c++20,89


<Figure size 1400x900 with 4 Axes>

## 3. Text cleaning that preserves technical meaning

برای C++ نباید پاک‌سازی aggressive باشد. توکن‌هایی مثل `std::vector`, `nullptr`, `template`, `gcc`, `cmake` برای شباهت مهم‌اند. بلوک‌های کد با نشانگر `CODE_BLOCK` حفظ و شمارش می‌شوند و عنوان دو بار تکرار می‌شود تا وزن بیشتری بگیرد.

In [5]:
from scripts.preprocess import strip_html_keep_code_markers, normalize_technical_text

sample = questions.iloc[0]
body_text, code_count = strip_html_keep_code_markers(sample['body_html'])
print('TITLE      :', sample['title'][:90])
print('clean title:', normalize_technical_text(sample['title'])[:90])
print('code blocks :', code_count)
print('clean body  :', normalize_technical_text(body_text)[:220])

TITLE      : How to use the C socket API in C++ on z/OS
clean title: how to use the c socket api in c++ on z os
code blocks : 9
clean body  : i m having issues getting the c sockets api to work properly in c++ on z os. although i am including code_block sys socket.h i still get compile time errors telling me that code_block af_inet is not defined. am i missing


## 4. Anti-leakage split and weak-relevance evaluation

- **train**: فیت vectorizerها و corpus کاندید
- **validation**: انتخاب وزن‌های ensemble
- **test**: فقط یک‌بار برای گزارش نهایی

relevance با هم‌پوشانی تگِ غیرعمومی تعریف می‌شود. جدول‌های زیر مستقیماً از خروجی‌های `pipeline.py` بارگذاری می‌شوند.

In [6]:
report = json.loads((REPORT_DIR / 'feature_engineering_report.json').read_text(encoding='utf-8'))
print('Split:', report['split'])
print('Best validation weights:', report['best_validation_weights'])

print('\nSingle-model validation metrics:')
display(pd.read_csv(REPORT_DIR / 'section2_single_model_validation_metrics.csv', index_col=0).round(4))

print('Top hybrid weight combinations (validation):')
display(pd.read_csv(REPORT_DIR / 'section2_hybrid_weight_search_validation.csv').head(5).round(4))

print('Final TEST metrics (evaluated once):')
display(pd.read_csv(REPORT_DIR / 'section2_best_hybrid_test_metrics.csv').T.rename(columns={0:'best_hybrid_test'}).round(4))

Split: {'type': 'temporal', 'train_rows': 1750, 'validation_rows': 375, 'test_rows': 375}
Best validation weights: {'title_word_tfidf': 0.22222222222222224, 'document_word_tfidf': 0.4444444444444445, 'char_wb_tfidf': 0.22222222222222224, 'lsa_document': 0.11111111111111112}

Single-model validation metrics:


,evaluated_queries,coverage,Hit@5,Recall@5,MAP@5,MRR@5,nDCG@5,Hit@10,Recall@10,MAP@10,MRR@10,nDCG@10,Hit@20,Recall@20,MAP@20,MRR@20,nDCG@20
model,,,,,,,,,,,,,,,,,
char_wb_tfidf,339,0.904,0.7404,0.0781,0.3088,0.5448,0.3936,0.8201,0.1288,0.2489,0.5557,0.3642,0.8909,0.1932,0.2004,0.5606,0.3374
document_word_tfidf,339,0.904,0.7021,0.0734,0.2799,0.5158,0.3623,0.8142,0.1288,0.2193,0.5306,0.3352,0.8997,0.1809,0.1745,0.5363,0.3096
lsa_document,339,0.904,0.5959,0.0516,0.2355,0.4228,0.3015,0.7168,0.0874,0.1952,0.4393,0.2861,0.8142,0.1371,0.1613,0.4462,0.2725
title_word_tfidf,339,0.904,0.4631,0.0359,0.1211,0.2904,0.1802,0.5988,0.0581,0.0919,0.3081,0.1688,0.7375,0.0997,0.0781,0.3178,0.1695


Top hybrid weight combinations (validation):


,evaluated_queries,coverage,Hit@5,Recall@5,MAP@5,MRR@5,nDCG@5,Hit@10,Recall@10,MAP@10,...,nDCG@10,Hit@20,Recall@20,MAP@20,MRR@20,nDCG@20,title_word_tfidf,document_word_tfidf,char_wb_tfidf,lsa_document
0,339,0.904,0.7227,0.0835,0.3040,0.5455,0.3888,0.8171,0.1334,0.2450,...,0.3601,0.8791,0.1896,0.2019,0.5618,0.3375,0.2222,0.4444,0.2222,0.1111
1,339,0.904,0.7168,0.0825,0.3019,0.5358,0.3856,0.8171,0.1353,0.2448,...,0.3595,0.8820,0.1899,0.2011,0.5536,0.3364,0.2000,0.5000,0.2000,0.1000
2,339,0.904,0.7198,0.0795,0.2986,0.5307,0.3820,0.7994,0.1260,0.2415,...,0.3533,0.8761,0.1882,0.2000,0.5472,0.3338,0.2500,0.3750,0.2500,0.1250
3,339,0.904,0.7139,0.0777,0.2949,0.5226,0.3790,0.7935,0.1281,0.2399,...,0.3504,0.8643,0.1823,0.1963,0.5391,0.3279,0.1818,0.4545,0.1818,0.1818
4,339,0.904,0.7198,0.0785,0.2913,0.5256,0.3765,0.7935,0.1274,0.2394,...,0.3498,0.8643,0.1816,0.1957,0.5409,0.3271,0.2000,0.4000,0.2000,0.2000


Final TEST metrics (evaluated once):


,best_hybrid_test
evaluated_queries,323.0000
coverage,0.8613
Hit@5,0.7183
Recall@5,0.0872
MAP@5,0.3069
MRR@5,0.5550
nDCG@5,0.3930
Hit@10,0.8235
Recall@10,0.1294
MAP@10,0.2481


### Baseline comparison

برای اینکه نشان دهیم مدل واقعاً ارزش اضافه می‌کند، آن را با دو baseline بی‌اهمیت می‌سنجیم: ترتیب **تصادفی** و **محبوبیت** (همان پربازدیدترین/پرامتیازترین سؤال‌ها برای همه). انتظار داریم مدل hybrid به‌وضوح از هر دو بهتر باشد.

In [7]:
# Sanity check: does the model beat trivial baselines on the same held-out test set?
print('Baselines vs. hybrid model on TEST:')
display(pd.read_csv(REPORT_DIR / 'section2_baseline_vs_model_test_metrics.csv', index_col=0)
        [['Hit@5', 'Hit@10', 'MAP@10', 'MRR@10', 'nDCG@10']].round(4))

Baselines vs. hybrid model on TEST:


,Hit@5,Hit@10,MAP@10,MRR@10,nDCG@10
random,0.1269,0.2012,0.0077,0.0731,0.0237
popularity,0.1053,0.1331,0.0077,0.0477,0.0207
hybrid_model,0.7183,0.8235,0.2481,0.5691,0.3614


## 5. Feature columns: model inputs vs. audit-only

برای جلوگیری از فیچرهای تکراریِ به‌شدت همبسته، صراحتاً مشخص می‌کنیم چه چیزی ورودی مدل است و چه چیزی فقط برای ردگیری نگه داشته شده. همچنین تگ‌ها به‌صورت multi-hot به‌عنوان **متادیتای اختیاری** ذخیره می‌شوند، نه فیچر امتیازدهی (چون معیار ارزیابی هم بر پایه‌ی تگ است و استفاده‌ی همزمان leakage می‌سازد).

In [8]:
display(pd.Series({
    'model_text_input': report['model_text_input'],
    'models': report['models'],
    'audit_only_columns': report['audit_only_columns'],
    'tag_multihot_columns (metadata only)': report['tag_multihot_columns'],
}))
print('\nAnti-overfitting controls:')
for item in report['anti_overfit_controls']:
    print(' -', item)

model_text_input                        hybrid of TF-IDF (title word, document word, c...
models                                  [title_word_tfidf, document_word_tfidf, char_w...
audit_only_columns                      [view_count, answer_count, score, log_view_cou...
tag_multihot_columns (metadata only)    [tag_c, tag_c_11, tag_qt, tag_windows, tag_tem...
dtype: object


Anti-overfitting controls:
 - Temporal train/validation/test split
 - TF-IDF vectorizers fitted only on the training corpus
 - Hybrid weights selected only on validation
 - Test set evaluated only once
 - Generic c++ tag removed from weak relevance labels
 - Tags kept as weak labels / optional metadata, not as default scoring features


## 6. Demo: recommend similar questions

با artifact ذخیره‌شده (`best_hybrid_recommender.joblib`) یک سؤال نمونه می‌دهیم و Top-K سؤال مشابه از کل corpus برگردانده می‌شود.

In [9]:
import joblib
from scripts.feature_engineering import rank_hybrid, models_from_dicts

artifact = joblib.load(PROJECT_ROOT / 'data' / 'models' / 'best_hybrid_recommender.joblib')
models = models_from_dicts(artifact['models'])
corpus = artifact['questions'].reset_index(drop=True)

def recommend(title, body_html='', top_k=10):
    body_text, _ = strip_html_keep_code_markers(body_html)
    t = normalize_technical_text(title)
    b = normalize_technical_text(body_text)
    q = pd.DataFrame([{'title_clean': t, 'document_clean': (t + ' ' + t + ' ' + b).strip()}])
    idx = rank_hybrid(models, q, artifact['weights'], top_k=top_k)[0][:top_k]
    out = corpus.iloc[idx][['question_id', 'title', 'tags']].copy()
    out.insert(0, 'rank', range(1, len(out) + 1))
    return out

display(recommend('How to fix undefined reference error in C++ template class?',
                  '<p>I get an undefined reference when using a template class split between header and cpp.</p>'))

,rank,question_id,title,tags
696,1,37099605,undefined reference to cblas_sgemm,c++|cblas|linker-errors|matrix
1019,2,63338629,requires clause for a non-template member func...,c++|c++20|requires-clause|templates|visual-studio
884,3,52883700,Undefined reference to explicitly instantiated...,c++|clang++|explicit-instantiation|g++|templates
505,4,21346230,How to g++ a C++ program using boost/asio.hpp ...,boost|c++
604,5,29851056,undefined reference to omp_get_thread_num usin...,c++|cmake|makefile|openmp|unix
1831,6,79875536,How to fix &#39;incomplete type&#39; error in ...,c++|circular-dependency|incomplete-type|templates
1709,7,79860329,Undefined reference to string constructor with...,c++|c++17|compiler-bug|gcc
952,8,58454037,How to fix &#39;reference to type requires an ...,c++|c++11|unordered-map
879,9,52666791,"How do I have to configure Visual Studio Code,...",c++|undefined-reference|visual-studio-code
748,10,41194177,Undefined reference to symbol &#39;gzclose&#39;,c++|opencv|qt|qt-creator|undefined-reference


## 7. Conclusion

این نسخه‌ی Section 2 صرفاً چند feature نمی‌سازد؛ یک مسیر قابل‌دفاع می‌سازد:

- داده‌ی تمیز و model-ready با حفظ توکن‌های فنی.
- split زمانی برای جلوگیری از leakage.
- مقایسه‌ی چند نمایش متنی و یک ensemble وزنی، با tuning فقط روی validation.
- گزارش متریک‌های retrieval روی test (یک‌بار).
- artifact آماده‌ی inference برای فاز ۳.

همه‌ی این منطق در اسکریپت‌های ماژولار Section 3 پیاده شده، پس Section 2 و Section 3 کاملاً هم‌خوان‌اند.